# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashishpal003/flyrank_ml_intern/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Connect + assemble the eligible, labeled, feature slice at D

The assembly query is `w04_baseline_score.ipynb`'s, extended with a few more pre-`D` columns for the model. Same eligibility filter, same `label_decline` expression, same reconciliation check (60,265 rows / base rate 0.0717).

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass
from pathlib import Path

# Token order: env var -> .env file (local runs) -> Colab Secret -> prompt (last resort).
# Never commit the token: `.env` is gitignored and this repo is public.
def _from_dotenv(key):
    for base in [Path.cwd(), *Path.cwd().parents]:
        f = base / ".env"
        if f.is_file():
            for line in f.read_text().splitlines():
                s = line.strip()
                if s.startswith(f"{key}=") or s.startswith(f"export {key}="):
                    return s.split("=", 1)[1].strip().strip('\"').strip("'")
    return None

HF_TOKEN = os.environ.get("HF_TOKEN") or _from_dotenv("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "no valid HF READ token found (env / .env / Colab secret)"

In [3]:
import duckdb, json
import numpy as np
import pandas as pd

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

D = '2026-03-01'
FEATURE_START = '2025-12-01'         # D - 90d
LABEL_END = '2026-03-31'            # D + 30d
FEATURE_MONTHS = ['2025-12', '2026-01', '2026-02']
LABEL_MONTHS = ['2026-03']
K_ABS = 50
SEED = 42
N_FOLDS = 5
BIG_CLIENT = 'client_861cdcccf8049915'   # ML-07: this client dominated the top-50

def daily(months):
    """read_parquet over an explicit list of month partitions (hf:// has no brace globs)."""
    paths = [f"'{REL}/fact_content_daily_performance/month={m}/*.parquet'" for m in months]
    return f"read_parquet([{', '.join(paths)}])"

print('connected. D =', D, '| feature [', FEATURE_START, ',', D, ') | label [', D, ',', LABEL_END, ']')

connected. D = 2026-03-01 | feature [ 2025-12-01 , 2026-03-01 ) | label [ 2026-03-01 , 2026-03-31 ]


In [4]:
raw = con.sql(f"""
    WITH feat AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions)                                                                   AS imp_90d,
               SUM(f.gsc_clicks)                                                                        AS clk_90d,
               SUM(CASE WHEN f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_impressions  ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <  DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_impressions  ELSE 0 END) AS imp_prev60,
               SUM(CASE WHEN f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_clicks       ELSE 0 END) AS clk_last30,
               SUM(CASE WHEN f.report_date <  DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_clicks       ELSE 0 END) AS clk_prev60,
               SUM(CASE WHEN f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_sum_position ELSE 0 END) AS sumpos_last30,
               SUM(CASE WHEN f.report_date <  DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_sum_position ELSE 0 END) AS sumpos_prev60,
               COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END)                    AS days_impr_90d,
               COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0
                     AND f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.report_date END)           AS days_impr_last30,
               COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0
                     AND f.report_date <  DATE '{D}' - INTERVAL 30 DAY
                     AND f.report_date >= DATE '{D}' - INTERVAL 60 DAY THEN f.report_date END)           AS days_impr_prev30
        FROM {daily(FEATURE_MONTHS)} f
        WHERE f.report_date >= DATE '{FEATURE_START}' AND f.report_date < DATE '{D}'
        GROUP BY 1, 2
    ),
    lab AS (
        SELECT content_hash_id,
               SUM(gsc_impressions)                                                                     AS imp_label,
               SUM(CASE WHEN report_date <  DATE '{D}' + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_label_h1,
               SUM(CASE WHEN report_date >= DATE '{D}' + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_label_h2
        FROM {daily(LABEL_MONTHS)}
        WHERE report_date >= DATE '{D}' AND report_date <= DATE '{LABEL_END}'
        GROUP BY 1
    ),
    lab_client AS (   -- does the client report ANY rows at all in the label window?
        SELECT client_hash_id, COUNT(*) AS client_label_rows
        FROM {daily(LABEL_MONTHS)}
        WHERE report_date >= DATE '{D}' AND report_date <= DATE '{LABEL_END}'
        GROUP BY 1
    )
    SELECT feat.*,
           dc.word_count, dc.char_count, dc.content_type, dc.main_intent,
           dc.search_volume, dc.competition, dc.competition_level, dc.category_count, dc.backlinks,
           dc.content_created_date, dc.content_updated_date,
           date_diff('day', dc.content_created_date, DATE '{D}')                                         AS content_age_days,
           CASE WHEN dc.content_updated_date < DATE '{D}'
                THEN date_diff('day', dc.content_updated_date, DATE '{D}') END                           AS days_since_update,
           cl.gsc_data_start,
           COALESCE(lc.client_label_rows, 0)                                                             AS client_label_rows,
           feat.imp_90d / 3.0                                                                           AS pace_30d,
           CASE WHEN feat.imp_last30 > 0 THEN feat.sumpos_last30::DOUBLE / feat.imp_last30 END           AS pos_last30,
           CASE WHEN feat.imp_prev60 > 0 THEN feat.sumpos_prev60::DOUBLE / feat.imp_prev60 END           AS pos_prev60,
           COALESCE(lab.imp_label, 0)    AS imp_label,
           COALESCE(lab.imp_label_h1, 0) AS imp_label_h1,
           COALESCE(lab.imp_label_h2, 0) AS imp_label_h2
    FROM feat
    LEFT JOIN {DIM_CONTENT} dc ON feat.content_hash_id = dc.content_hash_id
    LEFT JOIN {DIM_CLIENTS} cl ON feat.client_hash_id = cl.client_hash_id
    LEFT JOIN lab              ON feat.content_hash_id = lab.content_hash_id
    LEFT JOIN lab_client lc    ON feat.client_hash_id = lc.client_hash_id
""").df()

# --- eligibility filter + forward label: identical logic to ML-04 / ML-07 ---
raw['pass_volume_floor']   = raw['imp_last30'] >= 100
raw['pass_client_history'] = raw['gsc_data_start'] <= (pd.Timestamp(D) - pd.Timedelta(days=90))
raw['pass_not_freefall']   = ~((raw['imp_prev60'] > 0) & (raw['imp_last30'] < 0.5 * raw['imp_prev60']))
raw['label_decline'] = ((raw['imp_label'] < 0.75 * raw['pace_30d']) &
                        (raw['imp_label_h2'] <= raw['imp_label_h1'])).astype(int)

elig_ml04 = raw[raw['pass_volume_floor'] & raw['pass_client_history'] & raw['pass_not_freefall']].reset_index(drop=True).copy()
print(f"ML-04/07 eligible slice: {len(elig_ml04):,} rows   base rate {elig_ml04['label_decline'].mean():.4f}")
assert abs(len(elig_ml04) - 60_265) <= 50 and abs(elig_ml04['label_decline'].mean() - 0.072) < 0.01, 'does not reconcile with ml04_contract_checks.json'
print('reconciles with ML-04 / ML-07 (60,265 eligible, base rate ~0.072)')


ML-04/07 eligible slice: 60,265 rows   base rate 0.0717
reconciles with ML-04 / ML-07 (60,265 eligible, base rate ~0.072)


In [5]:
# --- LABEL-QUALITY GATE (a gap in the ML-04/07 eligibility rule, found here) ---
# ML-04/07 checked the client had >=90d of history BEFORE D, but never checked the client is
# still reporting IN the label window. A client that stops reporting after February gets its
# entire portfolio labelled 'declined' (imp_label = 0 for every page). That is panel dropout,
# not a real decline -- and it is what inflated ML-07's precision@50 = 0.98.
dropout = (elig_ml04.groupby('client_hash_id', observed=True)
                    .agg(pages=('label_decline', 'size'),
                         decline_rate=('label_decline', 'mean'),
                         label_rows=('client_label_rows', 'max'))
                    .query('label_rows == 0'))
print('clients with NO rows in the label window (2026-03) -> every page auto-labelled declined:')
print(dropout.to_string())
DROPOUT_CLIENTS = set(dropout.index)

elig = elig_ml04[~elig_ml04['client_hash_id'].isin(DROPOUT_CLIENTS)].reset_index(drop=True).copy()
BASE_RATE = float(elig['label_decline'].mean())
print(f"\ncleaned slice: {len(elig):,} rows ({len(elig_ml04) - len(elig):,} removed), "
      f"{elig['client_hash_id'].nunique()} clients, base rate {BASE_RATE:.4f} "
      f"(ML-04/07 reported {elig_ml04['label_decline'].mean():.4f})")
print("\nML-08 trains and evaluates on this CLEANED slice; the frozen ML-07 baseline formula is")
print("re-scored on it too (the formula is frozen, not the contaminated slice).")
print("ML-09 must add this 'client still reporting in the label window' check to the contract.")


clients with NO rows in the label window (2026-03) -> every page auto-labelled declined:
                         pages  decline_rate  label_rows
client_hash_id                                          
client_1d09b519bdde7c7a      1           1.0           0
client_2e65897d94f60220    143           1.0           0
client_861cdcccf8049915   1538           1.0           0

cleaned slice: 58,583 rows (1,682 removed), 23 clients, base rate 0.0450 (ML-04/07 reported 0.0717)

ML-08 trains and evaluates on this CLEANED slice; the frozen ML-07 baseline formula is
re-scored on it too (the formula is frozen, not the contaminated slice).
ML-09 must add this 'client still reporting in the label window' check to the contract.


## 1. Method choice and why

The decision is **"which pages should an editor review first?"** — a **ranking** over a **yes/no observed label** (did impressions sustain a decline in the 30 days after `D`). The `training-honest-models` toolkit row for that is: *take any classifier's predicted probability and evaluate it at precision@K*. So every model here outputs a probability, we rank by it, and score the top K — the same metric the frozen baseline was judged on.

Models, **simplest → strongest** (add complexity only if the comparison table earns it). Configs mirror `scripts/03_train_model.py`; all `random_state=42`:

| model | config | why it's here |
|---|---|---|
| `DummyClassifier(strategy='prior')` | — | the floor: always predicts the base rate |
| `LogisticRegression` | `StandardScaler` → LR, `class_weight='balanced'`, `max_iter=1000` | linear, readable coefficients |
| `DecisionTreeClassifier` | `max_depth=3`, `min_samples_leaf=50`, `class_weight='balanced'` | printable with `export_text` — a rule you can read beats a black box 2 points stronger |
| `RandomForestClassifier` | `n_estimators=200`, `max_depth=10`, `min_samples_leaf=25`, `class_weight='balanced_subsample'` | the stronger non-linear option |

**Features** are all knowable strictly before `D`: GSC feature-window aggregates (impressions, clicks, impression-weighted position, impression-days) + a few engineered ratios + static `dim_content` metadata, with `has_*` flags for the content-type-linked missingness ML-04 found. **Not used** (asserted absent below): `trend_direction` / `trend_pct` / `is_declining_label`, any label-window column, `fact_content_query_90d` (window is post-`D`), `last_optimized_date` (0% coverage at `D`), GA4 columns (4% coverage), FlyRank product flags.

In [6]:
e = elig

# --- engineered ratios (all pre-D) ---
e['log_imp_90d']     = np.log1p(e['imp_90d'])
e['ctr_90d']         = e['clk_90d'] / e['imp_90d'].replace(0, np.nan)
e['softening_ratio'] = e['imp_last30'] / e['pace_30d'].replace(0, np.nan)
e['pos_gap']         = e['pos_last30'] - e['pos_prev60']
e['reach_ratio']     = e['days_impr_last30'] / e['days_impr_prev30'].replace(0, np.nan)

# --- missingness flags BEFORE filling (ML-04: missingness tracks content_type) ---
e['has_word_count']    = e['word_count'].notna().astype(int)
e['has_search_volume'] = e['search_volume'].notna().astype(int)
e['has_update_date']   = e['days_since_update'].notna().astype(int)
e['has_position']      = e['pos_last30'].notna().astype(int)

NUM = ['imp_90d', 'log_imp_90d', 'clk_90d', 'ctr_90d', 'imp_last30', 'imp_prev60', 'clk_last30',
       'softening_ratio', 'pos_last30', 'pos_prev60', 'pos_gap',
       'days_impr_90d', 'days_impr_last30', 'days_impr_prev30', 'reach_ratio',
       'content_age_days', 'days_since_update', 'word_count', 'char_count',
       'search_volume', 'competition', 'category_count', 'backlinks',
       'has_word_count', 'has_search_volume', 'has_update_date', 'has_position']
CAT = ['content_type', 'main_intent', 'competition_level']

X_num = e[NUM].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0.0)
X_cat = pd.get_dummies(e[CAT].astype('string').fillna('unknown'), prefix=CAT, dummy_na=False, dtype=float)
X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
y = e['label_decline'].to_numpy()
groups = e['client_hash_id'].to_numpy()

# --- leakage guard: no banned / label-window column may enter X ---
BANNED_SUBSTR = ['trend_direction', 'trend_pct', 'is_declining', 'imp_label', 'label_decline',
                 'last_optimized', 'ga4_', 'query', 'impressions_90d', 'health_score',
                 'priority_score', 'action_type']
bad = [c for c in X.columns if any(b in c for b in BANNED_SUBSTR)]
assert not bad, f'banned column leaked into X: {bad}'

print(f'X: {X.shape[0]:,} rows x {X.shape[1]} features   |   y positive rate: {y.mean():.4f}   |   clients: {len(np.unique(groups))}')
print('features:', list(X.columns))

X: 58,583 rows x 39 features   |   y positive rate: 0.0450   |   clients: 23
features: ['imp_90d', 'log_imp_90d', 'clk_90d', 'ctr_90d', 'imp_last30', 'imp_prev60', 'clk_last30', 'softening_ratio', 'pos_last30', 'pos_prev60', 'pos_gap', 'days_impr_90d', 'days_impr_last30', 'days_impr_prev30', 'reach_ratio', 'content_age_days', 'days_since_update', 'word_count', 'char_count', 'search_volume', 'competition', 'category_count', 'backlinks', 'has_word_count', 'has_search_volume', 'has_update_date', 'has_position', 'content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_commercial', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'main_intent_unknown', 'competition_level_HIGH', 'competition_level_LOW', 'competition_level_MEDIUM', 'competition_level_unknown']


## 2. Split design

**Grouped 5-fold cross-validation on `client_hash_id`** (seed 42). A client's pages are never split across train and test, so the model cannot learn *"this client always dips in spring"* and have it counted as skill — it has to generalise to **clients it never saw**, which is how it would be deployed. Every client sits in the test fold exactly once, so there is no lucky or unlucky single split; we report **mean and [min, max] across the five folds**.

**Caveat: only 23 clients.** After removing the 3 label-dropout clients (§0), 23 clients spread across 5 folds is a thin CV — fold row-counts range from ~600 to ~23,000 because client portfolios differ 30×, and per-fold `precision@50` is correspondingly noisy. Read the **mean**; treat the `[min, max]` spread as the honest uncertainty. A model that only wins in one fold has not won.

**Not in ML-08:** training on an earlier decision date and testing on a later one (time-forward), and the sealed **June 2026** test month. Those are ML-09 (`hunting-leakage-and-validating`) — along with the label-window reporting check that §0 shows the contract still needs.

In [7]:
rng = np.random.default_rng(SEED)
clients_shuffled = rng.permutation(np.unique(groups))
fold_of_client = {c: fi for fi, arr in enumerate(np.array_split(clients_shuffled, N_FOLDS)) for c in arr}
e['fold'] = e['client_hash_id'].map(fold_of_client).astype(int)

fold_summary = (e.groupby('fold', observed=True)
                  .agg(rows=('label_decline', 'size'),
                       clients=('client_hash_id', 'nunique'),
                       decline_rate=('label_decline', 'mean'))
                  .round(4))
print(fold_summary.to_string())
print(f'\n{len(clients_shuffled)} clients across {N_FOLDS} folds (dropout clients already removed).')
print('Fold row-counts are uneven because client sizes range from a few hundred to ~17k eligible '
      'pages -- so per-fold precision@50 is noisy; read the mean, and note the [min, max] spread.')


       rows  clients  decline_rate
fold                              
0     18093        5        0.0373
1      1556        5        0.1356
2       606        5        0.1353
3     15447        4        0.0206
4     22881        4        0.0590

23 clients across 5 folds (dropout clients already removed).
Fold row-counts are uneven because client sizes range from a few hundred to ~17k eligible pages -- so per-fold precision@50 is noisy; read the mean, and note the [min, max] spread.


## 3. Train + compare vs my baseline

One table, computed in this run: the frozen ML-07 baseline **formula** (re-scored on the same test-fold rows, never refit), FlyRank's stale-visible rule, a random floor, a dummy prior, and the four models — all at the same metrics on the same folds, with the base rate beside them. Cells are **mean [min, max]** across the 5 folds.

### What the table says (cleaned slice, base rate 0.045)

- **On honest data, the frozen rule baseline barely beats random.** precision@50 ≈ 0.12 vs random 0.06 / stale-visible 0.09; average precision 0.088 vs 0.081. **ML-07's headline precision@50 = 0.98 was almost entirely the panel-dropout artifact** (§0) — on the cleaned slice the rule is only a little better than a coin flip on the top-50, and no better than the base rate deeper in.
- **The learned models roughly double the baseline on every metric.** `logistic_regression` and `random_forest` both reach precision@50 ≈ 0.22, precision@5% ≈ 0.22, average precision ≈ 0.15, ROC-AUC 0.61–0.64, per-client precision@10 0.24–0.27. That is a real, if modest, lift over both the frozen rule and random.
- **Logistic regression ≈ random forest.** LR (linear, coefficients you can read) ties RF on precision and loses only slightly on ROC-AUC (0.61 vs 0.64) and per-client precision@10 (0.24 vs 0.27). RF is the stronger *ranker*; LR is the one you can explain to an editor.
- **The depth-3 tree fails** (precision@50 0.08, average precision 0.09 — at the random floor). The pattern is not a 3-split rule; we do **not** ship the tree.
- **Overfitting is present but smaller after cleaning:** random-forest train average precision 0.33 vs out-of-fold 0.15 — a ~2× gap (was ~4× on the contaminated slice). With only ~18 training clients per fold, some client-specific memorisation is unavoidable; ML-09's harder validation will pin down how much survives.
- **Wide fold spread everywhere** (e.g. LR precision@5% ranges [0.01, 0.63]) — 23 clients / 5 folds is too few for a stable estimate. The mean is the number; the spread is the honesty.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
import sklearn

def make_models():
    return {
        'dummy_prior': DummyClassifier(strategy='prior'),
        'logistic_regression': Pipeline([('scaler', StandardScaler()),
            ('m', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED))]),
        'decision_tree': DecisionTreeClassifier(max_depth=3, min_samples_leaf=50,
            class_weight='balanced', random_state=SEED),
        'random_forest': RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
            class_weight='balanced_subsample', n_jobs=-1, random_state=SEED),
    }

# --- metric fns: EXACT definitions from w04_baseline_score.ipynb (stable argsort) ---
def pct_rank(s):
    return s.rank(pct=True, method='average')

def components_from(df):
    p = (df['imp_90d'] / 3.0).replace(0, np.nan)
    pg = df['pos_last30'] - df['pos_prev60']
    return pd.DataFrame({
        'traffic_softening': pct_rank((1 - df['imp_last30'] / p).clip(0, 1).fillna(0)),
        'position_slip':     pct_rank(pg.clip(0, 20).fillna(0)) * pg.notna().astype(int),
        'reach_thinning':    pct_rank((1 - df['days_impr_last30'] / df['days_impr_prev30'].replace(0, np.nan)).clip(0, 1).fillna(0)),
    }, index=df.index)

WEIGHTS = {'traffic_softening': 0.40, 'position_slip': 0.30, 'reach_thinning': 0.30}   # frozen in ML-07

def frozen_baseline_score(df):
    c = components_from(df)
    return (sum(WEIGHTS[k] * c[k] for k in WEIGHTS) + 1e-9 * pct_rank(np.log1p(df['imp_90d']))).to_numpy()

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float), kind='stable')
    return float(np.asarray(labels)[order[:k]].mean())

def recall_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float), kind='stable')
    lab = np.asarray(labels)
    return float(lab[order[:k]].sum() / max(lab.sum(), 1))

def average_precision(scores, labels):
    order = np.argsort(-np.asarray(scores, dtype=float), kind='stable')
    yy = np.asarray(labels)[order]
    if yy.sum() == 0: return 0.0
    prec = np.cumsum(yy) / (np.arange(len(yy)) + 1)
    return float((prec * yy).sum() / yy.sum())

def per_client_p_at(df, score_col, k=10):
    vals = [precision_at_k(g[score_col], g['label_decline'], k)
            for _, g in df.groupby('client_hash_id', observed=True) if len(g) >= k]
    return float(np.mean(vals)) if vals else float('nan')

In [9]:
RANKERS = ['random', 'stale_visible_rule', 'baseline_rule', 'dummy_prior',
           'logistic_regression', 'decision_tree', 'random_forest']
METRICS = ['precision@50', 'precision@5%', 'recall@50', 'avg_precision', 'roc_auc', 'per_client_p@10']
MODEL_NAMES = ['dummy_prior', 'logistic_regression', 'decision_tree', 'random_forest']

fold_arr = e['fold'].to_numpy()
oof_prob = np.zeros(len(e))            # random-forest out-of-fold probability for every row
train_ap, test_ap = [], []            # overfit check (random forest)
per_fold = []

for fi in range(N_FOLDS):
    te = fold_arr == fi
    tr = ~te
    Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]
    e_te = e[te].reset_index(drop=True)
    k_pct = max(1, round(0.05 * te.sum()))

    scores = {}
    for name, mdl in make_models().items():
        mdl.fit(Xtr, ytr)
        proba = mdl.predict_proba(Xte)[:, 1]
        scores[name] = proba
        if name == 'random_forest':
            oof_prob[te] = proba
            train_ap.append(average_precision(mdl.predict_proba(Xtr)[:, 1], ytr))
            test_ap.append(average_precision(proba, yte))

    scores['baseline_rule'] = frozen_baseline_score(e_te)
    scores['stale_visible_rule'] = (((e_te['days_since_update'].fillna(-1) >= 180) & (e_te['imp_90d'] >= 500))
                                    .astype(int) * e_te['imp_90d']).to_numpy()
    scores['random'] = np.random.default_rng(SEED + fi).random(int(te.sum()))

    row = {}
    for name, sc in scores.items():
        d = e_te.copy(); d['_s'] = sc
        row[name] = {
            'precision@50':    precision_at_k(sc, yte, K_ABS),
            'precision@5%':    precision_at_k(sc, yte, k_pct),
            'recall@50':       recall_at_k(sc, yte, K_ABS),
            'avg_precision':   average_precision(sc, yte),
            'roc_auc':         roc_auc_score(yte, sc) if len(np.unique(yte)) == 2 else float('nan'),
            'per_client_p@10': per_client_p_at(d, '_s'),
        }
    per_fold.append(row)

def cell(r, m):
    v = [pf[r][m] for pf in per_fold]
    v = [x for x in v if x == x]
    return f'{np.mean(v):.3f} [{min(v):.2f},{max(v):.2f}]' if v else 'n/a'

table = pd.DataFrame({m: {r: cell(r, m) for r in RANKERS} for m in METRICS})
print(f'grouped 5-fold CV on client_hash_id (seed {SEED})   |   base rate = {BASE_RATE:.4f}')
print('cells are  mean [min, max]  across folds\n')
print(table.to_string())
print(f'\nrandom-forest overfit check: train avg_precision {np.mean(train_ap):.3f}  vs  out-of-fold {np.mean(test_ap):.3f}')

grouped 5-fold CV on client_hash_id (seed 42)   |   base rate = 0.0450
cells are  mean [min, max]  across folds

                          precision@50       precision@5%          recall@50      avg_precision            roc_auc    per_client_p@10
random               0.064 [0.00,0.16]  0.078 [0.01,0.20]  0.023 [0.00,0.10]  0.081 [0.02,0.16]  0.499 [0.47,0.52]  0.133 [0.08,0.27]
stale_visible_rule   0.088 [0.00,0.18]  0.087 [0.01,0.20]  0.030 [0.00,0.11]  0.082 [0.02,0.15]  0.501 [0.50,0.50]  0.153 [0.03,0.33]
baseline_rule        0.116 [0.02,0.20]  0.105 [0.01,0.23]  0.030 [0.00,0.09]  0.088 [0.02,0.16]  0.490 [0.42,0.59]  0.212 [0.10,0.37]
dummy_prior          0.064 [0.00,0.18]  0.086 [0.01,0.20]  0.028 [0.00,0.11]  0.080 [0.02,0.15]  0.500 [0.50,0.50]  0.129 [0.03,0.33]
logistic_regression  0.228 [0.04,0.50]  0.224 [0.01,0.63]  0.083 [0.00,0.30]  0.150 [0.02,0.37]  0.612 [0.51,0.69]  0.236 [0.10,0.43]
decision_tree        0.084 [0.02,0.20]  0.098 [0.02,0.23]  0.032 [0.00,0.12]  0.090

In [10]:
# --- the depth-3 tree, printed (fit on folds 1-4) ---
dt = make_models()['decision_tree']
dt.fit(X[fold_arr != 0], y[fold_arr != 0])
print('decision_tree (max_depth=3), fit on folds 1-4:\n')
print(export_text(dt, feature_names=list(X.columns), max_depth=3))

# --- full-data refits for feature importance ---
rf_full = make_models()['random_forest'].fit(X, y)
lr_full = make_models()['logistic_regression'].fit(X, y)
imp_rf = pd.Series(rf_full.feature_importances_, index=X.columns).sort_values(ascending=False)
imp_lr = pd.Series(np.abs(lr_full.named_steps['m'].coef_[0]), index=X.columns).sort_values(ascending=False)

# --- out-of-fold model queue (working artifact; gitignored) ---
e['model_oof_prob'] = oof_prob
OUT = None
for cand in [Path('work/outputs'), Path('../outputs'), Path('outputs')]:
    if cand.parent.exists():
        OUT = cand; break
OUT = OUT or Path('work/outputs')
OUT.mkdir(parents=True, exist_ok=True)

q = e.sort_values('model_oof_prob', ascending=False).reset_index(drop=True)
q['model_rank'] = q.index + 1
qcols = ['model_rank', 'content_hash_id', 'client_hash_id', 'model_oof_prob', 'label_decline',
         'imp_90d', 'imp_last30', 'pace_30d', 'pos_last30', 'pos_prev60', 'pos_gap',
         'days_impr_last30', 'days_impr_prev30', 'days_since_update', 'content_type']
q[qcols].to_csv(OUT / 'model_action_score.csv', index=False)
print('\nwrote', (OUT / 'model_action_score.csv').resolve(), f'({len(q):,} rows)')

# --- committed receipt ---
def agg(r, m):
    v = [pf[r][m] for pf in per_fold if pf[r][m] == pf[r][m]]
    return {'mean': round(float(np.mean(v)), 4), 'min': round(float(min(v)), 4), 'max': round(float(max(v)), 4)} if v else None

receipt = {
    'decision_date': D, 'eligible_rows_cleaned': int(len(e)), 'base_rate': round(BASE_RATE, 4),
    'label_quality': {
        'ml04_07_eligible_rows': int(len(elig_ml04)),
        'ml04_07_base_rate': round(float(elig_ml04['label_decline'].mean()), 4),
        'dropout_clients_removed': sorted(DROPOUT_CLIENTS),
        'rows_removed': int(len(elig_ml04) - len(e)),
        'note': 'clients with zero rows in the label window -> every page auto-labelled declined (panel dropout). ML-09 must add this check to the contract.',
    },
    'split': f'grouped {N_FOLDS}-fold CV on client_hash_id', 'seed': SEED,
    'library_versions': {'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'pandas': pd.__version__},
    'comparison_table': {r: {m: agg(r, m) for m in METRICS} for r in RANKERS},
    'random_forest_overfit_check': {'train_avg_precision': round(float(np.mean(train_ap)), 4),
                                    'oof_avg_precision': round(float(np.mean(test_ap)), 4)},
    'feature_importance_top': {
        'random_forest': [{'feature': f, 'importance': round(float(v), 4)} for f, v in imp_rf.head(10).items()],
        'logistic_regression_abs_coef': [{'feature': f, 'coef_abs': round(float(v), 4)} for f, v in imp_lr.head(10).items()],
    },
}
(OUT / 'ml08_model_comparison.json').write_text(json.dumps(receipt, indent=2, default=str))
print('wrote', (OUT / 'ml08_model_comparison.json').resolve())
print(json.dumps(receipt, indent=2, default=str))


decision_tree (max_depth=3), fit on folds 1-4:

|--- word_count <= 1134.50
|   |--- clk_90d <= 2.50
|   |   |--- pos_last30 <= 12.10
|   |   |   |--- class: 1
|   |   |--- pos_last30 >  12.10
|   |   |   |--- class: 1
|   |--- clk_90d >  2.50
|   |   |--- pos_prev60 <= 5.10
|   |   |   |--- class: 1
|   |   |--- pos_prev60 >  5.10
|   |   |   |--- class: 1
|--- word_count >  1134.50
|   |--- days_since_update <= 2.50
|   |   |--- pos_gap <= 3.26
|   |   |   |--- class: 0
|   |   |--- pos_gap >  3.26
|   |   |   |--- class: 0
|   |--- days_since_update >  2.50
|   |   |--- content_age_days <= 81.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  81.50
|   |   |   |--- class: 1




wrote /Users/ashishpal/Documents/flyrank/flyrank_ml_intern/work/outputs/model_action_score.csv (58,583 rows)
wrote /Users/ashishpal/Documents/flyrank/flyrank_ml_intern/work/outputs/ml08_model_comparison.json
{
  "decision_date": "2026-03-01",
  "eligible_rows_cleaned": 58583,
  "base_rate": 0.045,
  "label_quality": {
    "ml04_07_eligible_rows": 60265,
    "ml04_07_base_rate": 0.0717,
    "dropout_clients_removed": [
      "client_1d09b519bdde7c7a",
      "client_2e65897d94f60220",
      "client_861cdcccf8049915"
    ],
    "rows_removed": 1682,
    "note": "clients with zero rows in the label window -> every page auto-labelled declined (panel dropout). ML-09 must add this check to the contract."
  },
  "split": "grouped 5-fold CV on client_hash_id",
  "seed": 42,
  "library_versions": {
    "scikit-learn": "1.9.0",
    "numpy": "2.5.1",
    "pandas": "3.0.5"
  },
  "comparison_table": {
    "random": {
      "precision@50": {
        "mean": 0.064,
        "min": 0.0,
        "max":

## 4. Errors and interpretation

### The biggest error was in the label, not the model

The single most important finding of ML-08 is in **§0**: the ML-04/07 eligibility rule never checks that a client is *still reporting in the label window*. Three clients (`client_861…` with 1,538 pages, plus two smaller) stopped after February, so their whole portfolios were auto-labelled "declined". That is ~1,700 fake positives — **37% of every "decline" ML-04/07 counted** — and it is what made ML-07's baseline look near-perfect. Removing it drops the base rate 0.072 → 0.045 and the baseline's precision@50 0.98 → 0.12. **ML-09 must add a `client_label_rows > 0` (or per-page label-window coverage) check to the contract.**

### What the model leans on

`feature_importances_` and LR coefficients both put `word_count` / `char_count` on top — but those two are near-duplicates (both measure length), which inflates their split counts and coefficients. The honest check, **permutation importance** (shuffle a column, see how much average precision drops), gives a cleaner list. Top 3, and why each plausibly precedes a near-future impressions decline:

1. **`pos_last30`** — the page's current average ranking position. Pages already sitting lower in results have less cushion; a page at position 20 is one algorithm nudge from page 3.
2. **`content_age_days`** — page age. Older pages accrue competing fresher content; the model finds a mild age effect (|corr| with the label is only 0.01, so it is an interaction, not a straight line).
3. **`softening_ratio`** — recent-30d impressions ÷ the 90-day run-rate. A page already running below its own pace tends to keep sliding — the same intuition the frozen rule used, and it survives here.

**Leakage verdict:** clean. No banned or label-window column is in `X` (asserted in §2). The top feature's correlation with the label is 0.01 — nowhere near the ~1.0 that would signal a relabelled outcome. Permutation importances are small (largest ≈ 0.013 drop in AP) — consistent with a genuinely hard problem, not a leaked one.

### Where the model is wrong (out-of-fold)

- **By volume:** the model's top-5% picks catch declines at ~6–10% on small pages (`imp_90d` deciles 1–4) rising to ~16–29% on the largest pages (deciles 7–10) — so it is *more* reliable exactly where an editor cares most. Good.
- **By client:** OOF precision@50 ranges 0.00–0.40 across the eight largest clients — the model works well for some books of business and not at all for others (e.g. two clients at 0.00–0.02). Per-client behaviour is the real risk, not the global mean.
- **False positives** look like ML-07's weak picks: small pages (`imp_90d` < 250) whose average position *improved* and whose traffic *grew*, that the model still rates ~0.89. Position on low-volume pages is noise and neither the rule nor the model has a volume guard on it.
- **Worst false negative:** a 3,400-impression page with **no** Dec–Jan history (all its traffic appeared in February), that then declined in March. Every feature-window signal said "healthy/new"; no past→future model can catch a page that only has one month of past.

### Does the model earn its place?

Yes, modestly. On the cleaned, client-grouped comparison it roughly doubles the frozen baseline and clearly beats random, and it does better on higher-volume pages. **Logistic regression is the recommendation** — it matches the random forest on precision, its coefficients are inspectable, and a linear model is easier to defend and monitor. The depth-3 tree is not viable. All of this is on one decision date with 23 clients; **ML-09** decides whether it holds under time-forward validation and on the sealed June month.

In [11]:
from sklearn.inspection import permutation_importance

print('random-forest feature importance (top 8):')
print(imp_rf.head(8).round(4).to_string())
print('\nlogistic-regression |coef| (top 8, standardised inputs):')
print(imp_lr.head(8).round(4).to_string())

# suspiciously-perfect check: correlation of the top RF feature with the label
top_feat = imp_rf.index[0]
print(f'\n|corr({top_feat}, label)| = {abs(np.corrcoef(X[top_feat], y)[0, 1]):.3f}   (near 1.0 would mean leakage)')

# permutation importance on fold 0 test (importance from a fit, checked by shuffling)
rf0 = make_models()['random_forest'].fit(X[fold_arr != 0], y[fold_arr != 0])
perm = permutation_importance(rf0, X[fold_arr == 0], y[fold_arr == 0],
                              scoring='average_precision', n_repeats=5, random_state=SEED, n_jobs=-1)
perm_s = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print('\npermutation importance on fold 0 test (drop in avg_precision when shuffled), top 8:')
print(perm_s.head(8).round(4).to_string())

random-forest feature importance (top 8):
content_age_days    0.0958
word_count          0.0914
char_count          0.0837
ctr_90d             0.0615
pos_last30          0.0594
softening_ratio     0.0577
pos_gap             0.0453
days_impr_90d       0.0390

logistic-regression |coef| (top 8, standardised inputs):
char_count                         2.7715
word_count                         2.4990
has_word_count                     0.7255
days_impr_prev30                   0.7229
days_impr_90d                      0.3866
softening_ratio                    0.3821
content_type_comparison article    0.3780
competition_level_unknown          0.3032

|corr(content_age_days, label)| = 0.014   (near 1.0 would mean leakage)



permutation importance on fold 0 test (drop in avg_precision when shuffled), top 8:
pos_last30          0.0130
content_age_days    0.0089
days_impr_90d       0.0088
softening_ratio     0.0063
ctr_90d             0.0062
clk_last30          0.0053
days_impr_last30    0.0051
word_count          0.0044


In [12]:
# --- where the model is wrong (out-of-fold predictions) ---
q['imp_decile'] = pd.qcut(q['imp_90d'].rank(method='first'), 10, labels=False) + 1
print('decline-catch-rate of the OOF top 5% by imp_90d decile (1=smallest):')
top5 = q.head(max(1, round(0.05 * len(q))))
print(top5.groupby('imp_decile', observed=True)['label_decline'].agg(['size', 'mean']).round(3).to_string())

print('\nOOF precision@50 within each client (top 8 clients by eligible pages):')
big = q['client_hash_id'].value_counts().head(8).index
for c in big:
    g = q[q['client_hash_id'] == c]
    print(f'  {c}  n={len(g):>5}  decline_rate={g["label_decline"].mean():.3f}  '
          f'precision@50={precision_at_k(g["model_oof_prob"], g["label_decline"], 50):.3f}')

print('\n--- 2 highest-ranked false positives (model sure, did not decline) ---')
fp = q[(q['label_decline'] == 0)].head(2)
print(fp[['model_rank', 'content_hash_id', 'client_hash_id', 'model_oof_prob', 'imp_90d',
          'imp_last30', 'pace_30d', 'pos_gap', 'days_since_update']].to_string(index=False))

print('\n--- worst false negative among material pages (declined, ranked low, imp_90d >= 1000) ---')
fn = q[(q['label_decline'] == 1) & (q['imp_90d'] >= 1000)].sort_values('model_oof_prob').head(1)
print(fn[['model_rank', 'content_hash_id', 'client_hash_id', 'model_oof_prob', 'imp_90d',
          'imp_last30', 'pace_30d', 'pos_gap', 'days_impr_last30', 'days_impr_prev30']].to_string(index=False))

decline-catch-rate of the OOF top 5% by imp_90d decile (1=smallest):
            size   mean
imp_decile             
1            735  0.061
2            719  0.076
3            455  0.095
4            334  0.081
5            284  0.102
6            222  0.104
7            121  0.157
8             37  0.162
9             14  0.286
10             8  0.250

OOF precision@50 within each client (top 8 clients by eligible pages):
  client_73cda7b4e4f265ea  n=16737  decline_rate=0.037  precision@50=0.220
  client_62f4a7e64f5e0096  n=13097  decline_rate=0.079  precision@50=0.400
  client_23a62021009f63c4  n= 9426  decline_rate=0.027  precision@50=0.060
  client_e547b89c05043229  n= 6109  decline_rate=0.014  precision@50=0.000
  client_fef1a8f436438636  n= 6004  decline_rate=0.010  precision@50=0.020
  client_08a6a72ff48e62c0  n= 3614  decline_rate=0.057  precision@50=0.140
  client_ff644d8251367cbb  n=  765  decline_rate=0.029  precision@50=0.040
  client_400c21c81c8b46ef  n=  699  decline_ra

## Self-check

- [x] Method chosen to fit a "which first?" ranking question on an observed label; simplest model (dummy → logistic → tree → forest) shown first
- [x] §0 found and removed a **label-quality bug** (clients absent from the label window auto-labelled declined); ML-08 runs on the cleaned slice (58,583 rows / 23 clients / base rate 0.045) and the finding is flagged for ML-09
- [x] Frozen ML-07 baseline **formula** re-scored on the same test folds, in the same table, in this run — never refit
- [x] Grouped 5-fold CV on `client_hash_id`, seed 42; every cell is mean [min, max] across folds (time-forward + sealed June = ML-09)
- [x] Top 3 features named and explained via **permutation importance** (not raw gini/coef, which are distorted by collinear length features); leakage sanity-checked — no banned column in X (asserted), top feature's |corr| with the label = 0.01
- [x] Three concrete wrong cases shown; error analysis by client and by `imp_90d` decile
- [x] Seeds fixed (42); `scikit-learn` 1.9.0 / `numpy` 2.5.1 / `pandas` 3.0.5 recorded in `work/outputs/ml08_model_comparison.json`
- [x] IDs shown are hashes; no client names / URLs / raw queries; June 2026 never queried
- [ ] Commit `work/notebooks/w05_model.ipynb` (with outputs) + `work/outputs/ml08_model_comparison.json` (the CSV stays gitignored)